# Generalização: overfitting e underfitting

**Objetivo:** reproduzir a experiência do widget do site — ajustar polinômios de vários graus e ver o erro de treino cair enquanto o de teste forma um 'U'.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)

## Dados: função verdadeira + ruído

In [ ]:
x_tr = np.sort(np.random.uniform(0, 1, 15))
y_tr = np.sin(2*np.pi*x_tr) + np.random.normal(0, 0.25, x_tr.size)
x_te = np.sort(np.random.uniform(0, 1, 200))
y_te = np.sin(2*np.pi*x_te) + np.random.normal(0, 0.25, x_te.size)
print("treino:", x_tr.size, "| teste:", x_te.size)

## Ajustar polinômios de grau 1 a 12

In [ ]:
from sklearn.metrics import mean_squared_error

graus = list(range(1, 13))
err_tr, err_te = [], []
for g in graus:
    coef = np.polyfit(x_tr, y_tr, g)
    err_tr.append(mean_squared_error(y_tr, np.polyval(coef, x_tr)))
    err_te.append(mean_squared_error(y_te, np.polyval(coef, x_te)))

figura = go.Figure()
figura.add_trace(go.Scatter(x=graus, y=err_tr, mode="lines+markers",
                            line=dict(color=AZUL), name="treino"))
figura.add_trace(go.Scatter(x=graus, y=err_te, mode="lines+markers",
                            line=dict(color=VERMELHO), name="teste"))
figura.update_yaxes(type="log")
figura.update_layout(title="Erro de treino x teste (escala log)",
                     xaxis_title="grau do polinomio", yaxis_title="MSE",
                     height=360, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Visualizar três regimes

In [ ]:
from plotly.subplots import make_subplots
xx = np.linspace(0, 1, 300)
figura = make_subplots(rows=1, cols=3,
                       subplot_titles=("grau 1 — underfitting", "grau 4 — equilibrio", "grau 12 — overfitting"))
coluna = 1
for g in [1, 4, 12]:
    coef = np.polyfit(x_tr, y_tr, g)
    figura.add_trace(go.Scatter(x=x_tr, y=y_tr, mode="markers",
                                marker=dict(color=VERMELHO, size=6), showlegend=False), row=1, col=coluna)
    figura.add_trace(go.Scatter(x=xx, y=np.sin(2*np.pi*xx), mode="lines",
                                line=dict(color=VERDE, dash="dash"), showlegend=False), row=1, col=coluna)
    figura.add_trace(go.Scatter(x=xx, y=np.polyval(coef, xx), mode="lines",
                                line=dict(color=AZUL), showlegend=False), row=1, col=coluna)
    coluna += 1
figura.update_yaxes(range=[-2, 2])
figura.update_layout(height=320, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## Exercícios

**1.** Qual grau minimiza o erro de teste? Coincide com o do treino?

**2.** Aumente o treino para 150 pontos. O overfitting do grau 12 diminui?

In [ ]:
# @title Solução
print("grau otimo (teste):", graus[int(np.argmin(err_te))])
print("grau otimo (treino):", graus[int(np.argmin(err_tr))], "-> o treino sempre melhora com mais grau")
# Com mais dados, o polinomio de grau alto tem menos liberdade para se colar
# ao ruido: o overfitting diminui.